
---
title: "Bayesian Regression Explained: A Real-World Battery Degradation Case Study"
date: 2025-12-10
description: "Learn how Bayesian regression works in practice using PyMC, through a real-world
  battery degradation case study. This article shows how probabilistic modeling
  improves prediction, uncertainty quantification, and decision-making in
  high-stakes industrial systems."

image: posterior_sensitivity.svg
twitter-card: 
    image: "posterior_sensitivity.svg"
open-graph: 
    image: "posterior_sensitivity.svg"

categories:
  - python
  - bayesian

title-block-banner: "bayesion-01.jpg"
format:
  html:
    code-fold: true
    code-summary: "Show the code"
    code-overflow: wrap
    shift-heading-level-by: 1
    reference-location: margin
    quarto-template-params:
      banner-header-class: "blog-post"
---

> **Series**: *Bayesian Modelling for Industrial Applications* – **Part 2**  
> **Prerequisite**: [Part 1 – Understanding Bayesian Thinking](https://sambaiga.github.io/blog/2025/10/bayesian-modelling-01.html)

Predictive modeling in industrial settings is rarely just about accuracy. Decisions informed by models often carry **financial, safety, and operational risk**. In such environments, understanding *uncertainty* can be just as important as making a good point prediction. This article presents a practical introduction to **Bayesian regression** using a real-world case study: predicting lithium-ion battery degradation. Rather than treating model parameters as fixed but unknown values, Bayesian regression allows us to model uncertainty explicitly—giving engineers and data scientists a richer, more actionable understanding of system behavior.


This post is part of the **Bayesian Modelling for Industrial Applications** series. If you are new to Bayesian thinking, you may want to start with the first post in the series, where we introduce the foundational ideas behind Bayesian inference.


---

## Introduction

Welcome back to our series on **Bayesian Modelling for Industrial Applications**. In
[Part 1](https://sambaiga.github.io/blog/2025/10/bayesian-modelling-01.html), we explored how Bayesian thinking provides a principled framework for decision-making under uncertainty when evidence is limited. 

In this post, we extend that foundation to **continuous prediction problems**, showing how **Bayesian regression** transforms noisy industrial data into actionable insights—*with uncertainty explicitly quantified rather than ignored*.

Before diving into the mathematics, let’s frame the real-world challenge.

Industrial systems rarely produce clean or perfectly repeatable measurements. Sensor noise, unit-to-unit variability, and stochastic physical processes are the norm rather than the exception. However, traditional regression methods typically summarize all this complexity with a single “best-fit” curve.

Such point estimates overlook critical aspects of real industrial data, including:

- Measurement uncertainty inherent in sensors and data acquisition systems  
- Unit-to-unit variability in components (e.g., subtle differences between nominally identical batteries)  
- Random and nonlinear degradation behaviour  
- Limited early-life observations, a common constraint in industrial testing  

Bayesian regression addresses these challenges by treating model parameters as **probability distributions** rather than fixed values. This allows us to quantify uncertainty, incorporate prior engineering knowledge, and make **risk-aware predictions**—capabilities that are essential in domains such as battery manufacturing, aerospace systems, and reliability engineering.


### Why Bayesian Regression?

Imagine managing a fleet of electric vehicles. One of your biggest challenges is predicting
**battery State of Health (SoH)** over time. Early predictions inform warranty decisions,
maintenance planning, and safety margins.

A purely deterministic model might produce a single degradation curve. In practice, however,
real batteries age differently—even under similar conditions.

Bayesian regression addresses this reality by treating uncertainty as a first-class citizen.
Instead of delivering a single prediction, it provides **ranges of plausible outcomes**,
allowing decisions to be made with risk explicitly accounted for.

### The Foundation: Modeling Distributions with Bayes' Theorem

Bayesian regression models uncertainty by treating parameters as probability distributions
rather than fixed values. Each coefficient represents a range of plausible effects, informed
by both prior knowledge and observed data.

This framework is built on three core components:

1. **Priors**, which encode existing engineering knowledge or physical constraints  
2. **Likelihood**, which links the model to noisy real-world measurements  
3. **Posterior**, which combines prior information and data into an updated belief

The posterior distribution enables **credible intervals**, allowing us to quantify how
confident we are in both parameter estimates and future predictions—an essential capability
in industrial decision-making.

## Case Study: Predicting Battery Degradation with CALCE Data

Lithium-ion batteries are critical components in electric vehicles and stationary energy
storage systems. Unexpected capacity loss can lead to service interruptions, safety risks,
and costly premature replacements.

The challenge is predicting future battery health when:
- Direct capacity measurements are infrequent and expensive
- Early-life data is sparse
- Degradation accelerates nonlinearly near end of life

The gaol is not only to predict degradation, but to quantify uncertainty well enough to
support maintenance and replacement decisions.


We use battery degradation data from the  [CALCE Battery](https://calce.umd.edu/battery-data) Research Group at the University of Maryland. Battery State of Health ($\text{SoH}$) is defined as current capacity relative to initial capacity ($\text{SoH} = C/C_0$).  This continuous value degrades non-linearly over the battery's life, driven primarily by cycle count and operational conditions, with significant measurement noise. The [CALCE dataset](https://calce.umd.edu/battery-data) provides over 1,200 capacity measurements   taken at discrete cycle intervals, alongside features like charging and discharge current and voltage. This comprehensive dataset has become a benchmark in battery research, allowing rigorous comparison of degradation models

In [ ]:
import arviz as az
from great_tables import GT, loc, md, style
from IPython.display import clear_output
from lets_plot import (
    LetsPlot,
    aes,
    coord_cartesian,
    facet_wrap,
    flavor_high_contrast_dark,
    geom_area,
    geom_band,
    geom_density,
    geom_histogram,
    geom_line,
    geom_point,
    geom_ribbon,
    geom_vline,
    gggrid,
    ggplot,
    ggsize,
    guide_legend,
    guides,
    labs,
    layer_tooltips,
    scale_color_brewer,
    scale_color_manual,
    scale_fill_manual,
    scale_y_continuous,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor as pt
from sklearn.preprocessing import MinMaxScaler, StandardScaler

az.style.use("arviz-doc")

from bayes.plot.basic_plots import line_plot, modern_theme, pro_colors, scatter_plot

LetsPlot.setup_html(isolated_frame=False, offline=True, no_js=True, show_status=False)
np.random.seed(42)

### 2.1 Choosing the Beta Likelihood


The capacity data being modeled, $C$, represents the battery's health and is strictly bounded between zero and its initial (maximum) capacity, $C_{\text{max}}$. The goal of this analysis is to model the evolution of $C$. In many standard regression approaches, the model's likelihood function (which defines the distribution of the noise) is assumed to be Gaussian (Normal). This assumption is fundamentally incompatible with the physical reality of capacity degradation for two key reasons.

1.  Gaussian models assume the target variable can take any real value ($-\infty$ to $+\infty$), ignoring the fact that the underlying capacity $C$ and the derived $\text{SoH}$ cannot fall outside their physical limits (i.e., $[0, C_{0}]$ or $[0, 1]$)
2.  With enough extrapolation, Gaussian models produce impossible values (e.g.,negative capacity). In safety-critical systems, such predictions are dangerous.
   
Since battery capacity degradation is continuous and bounded between 0 and 1, a Beta likelihood provides a natural modeling choice avoiding ad-hoc truncation required by Gaussian assumptions. The Beta distribution is defined by two shape parameters ($\alpha$ and $\beta$) such that:$$\text{SoH}_i \sim \text{Beta}(\alpha_i, \beta_i)$$ 

```python
with pm.Model() as battery_model:
    pm.Beta("y_obs", alpha=alpha, beta=beta_shape, observed=y_data)
```
where $\alpha_i, \beta_i > 0$ and $$\text{SoH}_i=\frac{C_i}{C_0} \in [0, 1]$$

The Beta distribution is flexible, capable of modeling various shapes (uniform, U-shaped, skewed) depending on the values of $\alpha$ and $\beta$. This modeling choice ensures all predicted $\text{SoH}$ values remain physically plausible, while also allowing flexible modeling of various degradation patterns through the shape parameters.

#### Parameterization: Mean ($\mu$) and Precision ($\phi$)

To make the parameters intuitive, the Beta distribution is typically reparameterized using the mean ($\mu$) and the precision ($\phi$).The shape parameters, $\alpha_i$ and $\beta_i$, which define the exact shape of the distribution for a given observation, are calculated directly from the mean $\mu_{i} \in (0, 1)$ and the global precision $\phi > 0$ such that:$$\alpha_i = \mu_{i} \cdot \phi \quad \text{and} \quad \beta_i = (1 - \mu_{i}) \cdot \phi$$ The precision parameter, $\phi$, controls the variance: a large $\phi$ means the predictions are tightly clustered around the mean $\mu_{i}$, indicating low uncertainty (low variance).


```python
with pm.Model() as battery_model:
    alpha = mu_scaled * phi
    beta_shape = (1 - mu_scaled) * phi
```

The mean parameter $\mu_{i} \in (0, 1)$ must be linked to our predictors. Since the mean is bounded by $(0, 1)$, we use the Logit Link Function to map the linear combination of predictors ($\eta_i$) to this interval: $$\text{logit}(\mu_{i}) = \eta_i$$ as such , $$\mu_{i} = \text{logit}^{-1}(\eta_i) = \frac{1}{1 + e^{-\eta_i}}$$

```python
with pm.Model() as battery_model:
    mu_scaled = pm.Deterministic("mu_scaled", pm.math.sigmoid(logit_mu))
```

> 💡 Key Takeaway: The Logit Link function is the mathematical bridge that ensures our mean prediction, $\mu_i$, respects the physical boundary of $(0, 1)$ imposed by the Beta distribution

#### The Linear Predictor: Capturing Degradation

The core of our predictive power lies in the linear predictor, $\eta_i$. It is structured to incorporate both the fundamental, non-linear degradation due to cycling and the linear operational effects from features $\mathbf{x}$ like voltage and current:
$$\eta_i = \underbrace{\beta_0}_{\text{Intercept}} + \underbrace{f(k_i)}_{\text{Non-linear Decay}} + \underbrace{\mathbf{x}_i^{\top} \boldsymbol{\beta}}_{\text{Operational Effects}}$$

In this work, the non-linear decay component is modeled as an exponential function: $$f(k_i) = -A\cdot(1-e^{-\lambda \cdot k_i})$$ where $k_i$ is the cycle count, $\lambda$ is the degradation rate, and $A>0$ is a learnable amplitude parameter controlling the strength of decay. This captures the physical reality that battery capacity degrades rapidly at first and then more slowly over time. where:


```python
with pm.Model() as battery_model:
    degradation = pm.math.exp(-lambda_rate * cycle_data)
    degradation_term = -degr_amp * (1 - degradation)
    logit_mu = intercept + degradation_term + pm.math.dot(x_data, beta)
```

The components of $\eta_i$ are as follows:


1. Intercept ($\beta_0$): The baseline capacity on the logit scale when operational effects are zero and the cycle count ($k_i$) is zero.
2. Degradation Term ($e^{-\lambda \cdot k_i}$): This is the non-linear exponential decay over the cycle count $k_i$, controlled by the rate $\lambda$. This term ensures the capacity prediction naturally trends downward toward zero capacity over time.
3. Operational Effects ($\mathbf{x}_{i}^{\top} \boldsymbol{\beta}$): This is a standard linear combination, where $\boldsymbol{\beta}$ is the vector of coefficients for the standardized operational features $\mathbf{x}_{i}$. 
   
This models how factors like maximum temperature accelerate or slow down the degradation.

> 🧠 **Self-Test**: You are modeling $\text{SoH}$, which must stay in $[0, 1]$. Your linear predictor, $\eta_i = \beta_0 + \mathbf{x}_i^{\top} \boldsymbol{\beta}$, can produce values ranging from $-\infty$ to $+\infty$.What would happen if you skipped the Logit Link Function and simply set $\mu_i = \eta_i$? Why is the Logit Link function mandatory for the Beta regression model?

### Encoding Knowledge with Priors

In Bayesian modeling, defining priors is a critical step. This step allows domain knowledge accumulated from battery engineering to be embedded directly into the model, ensuring that predictions remain physically plausible even when data is sparse. A prior distribution is assigned to every unknown parameter ($\beta_0, \boldsymbol{\beta}, \lambda, \phi$). These priors act as soft constraints, preventing the model from learning extreme or non-physical relationships.

**The Intercept ($\beta_0$)** 

The Intercept $\beta_0$ represents the initial capacity of the battery fleet on the logit scale. The orange curve in the figure below represents the selected informative prior, $\text{Normal}(\mu_{\text{logit\_start}}, 0.5^2)$. A standard deviation of $\sigma = 0.5$ is chosen to balance prior knowledge (centering at $\mu_{\text{logit\_start}}$) with sufficient uncertainty to allow the observed data to meaningfully influence the final estimate.

```python
with pm.Model() as battery_model:
    eps=1e-8
    initial_logit_capacity_mean = -np.log(1-eps)
    intercept = pm.Normal("intercept", mu=initial_logit_capacity_mean, sigma=0.5)
```

In [ ]:
from bayes.plot.distribution import plot_density

In [ ]:
n = 1000
s1 = pm.draw(pm.Normal.dist(mu=0.28, sigma=0.1), n)
s2 = pm.draw(pm.Normal.dist(mu=0.28, sigma=0.2), n)
s3 = pm.draw(pm.Normal.dist(mu=0.28, sigma=0.5), n)

df = pd.DataFrame(
    {
        "value": np.concatenate([s1, s2, s3]),
        "distribution": np.repeat(["(μ=0.28, σ=0.1)", "(μ=0.28, σ=0.2)", "(μ=0.28, σ=0.5)"], n),
    }
)

plot_density(df, title="Normal Distributions Intercept Priors", fig_size=(500, 400))

The narrower blue ($\sigma = 0.1$) and green ($\sigma = 0.2$) curves represent highly concentrated priors that would strongly restrict the posterior estimates. The wider $\sigma = 0.5$ (orange) distribution corresponds to a more conservative informative prior, granting the initial capacity estimate $\beta_0$ a reasonable degree of uncertainty.

**Operational Effects ($\boldsymbol{\beta}$)**

The vector of coefficients $\boldsymbol{\beta}$ controls the influence of operational features on capacity fade. Engineering knowledge suggests that, unless a feature is extreme, its immediate effect on capacity should be subtle, as the overall degradation process is primarily driven by cycle count.

```python
    with pm.Model() as battery_model:
    beta = pm.Normal("beta", mu=0, sigma=0.2, shape=n_features)
``` 


In [ ]:
from bayes.plot.distribution import plot_density

In [ ]:
n = 1000
s1 = pm.draw(pm.Normal.dist(mu=0, sigma=0.1), n)
s2 = pm.draw(pm.Normal.dist(mu=0, sigma=0.2), n)
s3 = pm.draw(pm.Normal.dist(mu=0, sigma=1.0), n)

df = pd.DataFrame(
    {
        "value": np.concatenate([s1, s2, s3]),
        "distribution": np.repeat(["(μ=0, σ=0.1)", "(μ=0, σ=0.2)", "(μ=0, σ=1.0)"], n),
    }
)

plot_density(df, title="Normal Distributions Beta Priors", fig_size=(500, 400))

As shown in the figure above, a tight informative prior, $\text{Normal}(0, 0.2^2)$, is used for $\boldsymbol{\beta}$. Centering this prior at zero reflects the assumption that, on average, operational features have no effect, while the small standard deviation ($0.2$) requires strong evidence from the data before attributing a large effect to any single feature. This constraint prevents non-physical, abrupt changes in capacity predictions. In contrast, a broader prior such as $\text{Normal}(0, 1.0^2)$ (orange curve) allows extreme effects that are considered non-physical.


**Degradation rate $\lambda$** 

The degradation rate $\lambda$ governs the exponential decay term $e^{-\lambda k_i}$. Since degradation must always occur and capacity cannot increase indefinitely, it is necessary to enforce $\lambda > 0$. Accordingly, a Log-Normal prior, $\text{LogNormal}(\ln(0.005), 0.5^2)$, is used for $\lambda$.

```python
with pm.Model() as battery_model:
    lambda_rate = pm.Lognormal("lambda_rate", mu=np.log(0.01), sigma=0.5)
```
> 🧠 **Self-Test**: Recall that we set the prior for the fade rate $\lambda$ as $\text{LogNormal}(\ln(0.01), 0.5^2)$ (where $\sigma = 0.5$). What practical problem would arise if an engineer, overly confident in their historical knowledge, reset the prior to $\text{LogNormal}(\ln(0.01), 0.1^2)$ (where $\sigma = 0.1$)?

In [ ]:
n = 1000
s1 = pm.draw(pm.LogNormal.dist(np.log(0.005), sigma=0.1), n)
s2 = pm.draw(pm.LogNormal.dist(np.log(0.005), sigma=0.5), n)
s3 = pm.draw(pm.LogNormal.dist(np.log(0.005), sigma=1.0), n)

df = pd.DataFrame(
    {
        "value": np.concatenate([s1, s2, s3]),
        "distribution": np.repeat(["(μ=In(0.005), σ=0.1)", "(μ=In(0.005), σ=0.5)", "(μ=In(0.005), σ=1.0)"], n),
    }
)

plot_density(df, title="LogNormal Distributions Priors", fig_size=(500, 400))

This weakly informative prior centers the expected degradation rate around $\mathbf{0.5\%}$, while the spread $\sigma = 0.5$ (green/teal curve) is sufficiently wide to accommodate realistic fleet-level variability. At the same time, it remains substantially tighter than $\sigma = 1.0$ (orange curve), thereby avoiding non-physical probability mass assigned to unrealistically large degradation rates.

This distribution reflects a conservative estimate of uncertainty, allowing greater variation in degradation behavior than a tighter prior (e.g., $\sigma = 0.1$) would permit, while still preventing implausible rates.

**Degradation Amplitude ($A$)**

The parameter degr_amp ($A$) controls the overall amplitude of the degradation component. Since this amplitude must be non-negative, a Half-Normal distribution is used, which has support only on positive values. The scale parameter $\sigma$ determines the strength of regularization.


```python
   with pm.Model() as battery_model:
    degr_amp = pm.HalfNormal("degr_amp", sigma=0.1)
```
 

In [ ]:
n = 1000
s1 = pm.draw(pm.HalfNormal.dist(sigma=0.1), n)
s2 = pm.draw(pm.HalfNormal.dist(sigma=0.2), n)
s3 = pm.draw(pm.HalfNormal.dist(sigma=0.5), n)


df = pd.DataFrame(
    {
        "value": np.concatenate([s1, s2, s3]),
        "distribution": np.repeat(["σ=0.1", "σ=0.2", "σ=0.5"], n),
    }
)
plot_density(df, title="Gamma Distributions Phi Priors", fig_size=(500, 400))

As shown in the figure above, $\text{HalfNormal}(\sigma = 0.1)$ strongly concentrates probability mass near zero, requiring substantial evidence before attributing a large degradation amplitude. In contrast, broader priors such as $\text{HalfNormal}(\sigma = 0.5)$ place non-negligible probability on large, non-subtle amplitudes (up to approximately $1.0$), increasing the risk of overfitting by allowing the model to explain noise through the amplitude term.

**Precision Parameter ($\phi$)**

The precision parameter $\phi$ controls the variance of the Beta likelihood and represents the expected level of noise in the $\text{SoH}$ measurements. Accordingly, a highly informative Gamma prior, $\text{Gamma}(100, 2)$, is assigned to $\phi$.
```python
with pm.Model() as battery_model:
    phi = pm.Gamma("phi", alpha=100, beta=2.0)
```
This prior is centered at $\mathbb{E}[\phi] = \alpha / \beta = 50$ with a relatively small standard deviation ($\sigma_{\phi} = 5.0$), indicating high confidence in this expectation. This choice encodes the belief that sensor noise is low ($\sigma_{\text{noise}} \approx 0.14$), reflecting the physical reality of precise laboratory-grade measurements.

In [ ]:
n = 1000
s1 = pm.draw(pm.Gamma.dist(alpha=10, beta=1), n)
s2 = pm.draw(pm.Gamma.dist(alpha=50, beta=5), n)
s3 = pm.draw(pm.Gamma.dist(alpha=100, beta=2), n)


df = pd.DataFrame(
    {
        "value": np.concatenate([s1, s2, s3]),
        "distribution": np.repeat(["Gamma(α=10, β=1)", "Gamma(α=50, β=5)", "Gamma(α=100, β=2)"], n),
    }
)
plot_density(df, title="Gamma Distributions Phi Priors", fig_size=(500, 400))


From the figure above, it is evident that $\text{Gamma}(\alpha = 100, \beta = 2.0)$ (orange curve) provides a strong belief in high precision. In contrast, $\text{Gamma}(\alpha = 10, \beta = 1.0)$ yields a lower expected precision with greater spread, allowing excessive uncertainty and risking a flat, unphysical prior predictive distribution. Alternative Gamma priors with the same expected precision but larger variance similarly underestimate the precision of modern sensors.

The complete model now combines all these components:
    

In [ ]:
def beta_regression_model(
    data: pd.DataFrame,
    features: list[str],
    target: str = "capacity",
    scaler: StandardScaler | None = None,
    lower_bound: float = 0.2,
    upper_bound: float = 1.3,
    eps: float = 1e-8,
) -> tuple[pm.Model, StandardScaler]:
    """Beta regression model for bounded battery capacity data using PyMC.

    Capacity (SoH) is scaled to the (0, 1) interval for the Beta distribution.

    Args:
        data: DataFrame containing 'capacity', 'cycle', and feature columns.
        features: List of column names used as predictors (X variables).
        target: Name of the capacity column.
        scaler: Pre-fitted StandardScaler object, or None to fit a new one.
        lower_bound: Physical lower bound for capacity (for scaling).
        upper_bound: Physical upper bound for capacity (for scaling).
        eps: Small value to avoid boundary issues in Beta distribution.

    Returns:
        A tuple containing the PyMC model and the fitted/provided StandardScaler.
    """
    # 1. Prepare Features (X)
    if scaler is None:
        scaler = StandardScaler()
        x_scaled = scaler.fit_transform(data[features])
    else:
        x_scaled = scaler.transform(data[features])

    # 2. Prepare Targets (Y)
    y = data[target].values.astype(np.float64)
    cycles = data["cycle"].values.astype(np.float64)
    n_features = len(features)

    # Transform y to (0,1) interval and clip to avoid boundaries (0 or 1)
    y_scaled = (y - lower_bound) / (upper_bound - lower_bound)
    y_scaled = np.clip(y_scaled, eps, 1 - eps)

    with pm.Model() as model:
        # Data Containers
        x_data = pm.Data("x_data", x_scaled)
        cycle_data = pm.Data("cycle_data", cycles)
        y_data = pm.Data("y_data", y_scaled)

        # Priors
        initial_logit_capacity_mean = -np.log(1 - 1e-6)
        intercept = pm.Normal("intercept", mu=initial_logit_capacity_mean, sigma=0.5)
        lambda_rate = pm.Lognormal("lambda_rate", mu=np.log(0.005), sigma=0.5)
        beta = pm.Normal("beta", mu=0, sigma=0.2, shape=n_features)
        phi = pm.Gamma("phi", alpha=100, beta=2.0)
        degr_amp = pm.HalfNormal("degr_amp", sigma=0.1)

        # Linear predictor (eta) on logit scale
        degradation = pm.math.exp(-lambda_rate * cycle_data)
        degradation_term = -degr_amp * (1 - degradation)
        logit_mu = intercept + degradation_term + pm.math.dot(x_data, beta)

        # Convert to probability scale (0,1)
        mu_scaled = pm.Deterministic("mu_scaled", pm.math.invlogit(logit_mu))

        # Beta likelihood
        alpha = mu_scaled * phi
        beta_shape = (1 - mu_scaled) * phi
        pm.Beta("y_obs", alpha=alpha, beta=beta_shape, observed=y_data)

        # Transform mu back to original scale
        mu_original = pm.Deterministic("mu_original", mu_scaled * (upper_bound - lower_bound) + lower_bound)

        pm.Deterministic("capacity_pred", mu_original)
        pm.Deterministic("feature_effects", beta)

    return model, scaler

## Feature Engineering: Translating Raw Data to Diagnostics

In the preceding sections, the output side of the Bayesian model was rigorously defined, including the Beta likelihood, the Logit link function, and physics-informed priors for the parameters ($\beta_0, \boldsymbol{\beta}, \lambda, \phi$). However, the quality of the resulting predictions depends critically on the quality of the input features ($\mathbf{x}$) that drive the degradation term ($\eta_i = \dots + \mathbf{x}_i^{\top} \boldsymbol{\beta}$).

Raw capacity measurement curves are noisy and variable. Therefore, before proceeding to Bayesian sampling, it is necessary to dedicate a structured process to translating real-world operational data into robust, physically meaningful diagnostic features.

This motivates the crucial step of feature engineering.

### Data Alignment and Cleaning

Before extracting diagnostic features, a uniform time base must be established, as the formulas used for feature extraction require comparable voltage and current values across cycles.

1. Cycle Alignment (Standardization): Linear interpolation is used to resample all voltage and current time-series arrays to a uniform length (e.g., 500 points). This standardization enables direct cycle-to-cycle comparison, as illustrated by the transition from the raw data (Figures 1 and 2) to the interpolated curves (Figures 3 and 4).

<div style="display: flex; flex-wrap: wrap; gap: 20px; justify-content: center; margin: 30px 0;"> <div style="flex: 1; min-width: 300px; max-width: 600px; text-align: center;"> <img src="charge_voltage.png" alt="Cycle 1" style="width: 100%; height: auto; border-radius: 8px;" /> <p style="margin-top: 12px; font-size: 15px; color: #555;"> Figure 1: Charging Voltage curve at the beginning of life </p> </div>
As shown in Figures 1 and 2, voltage curves differ in length due to variations in charge and discharge durations. Linear interpolation standardizes these curves to a fixed length (e.g., 500 points), enabling direct comparison across cycles, as illustrated in Figures 3 and 4.

<div style="flex: 1; min-width: 300px; max-width: 600px; text-align: center;"> <img src="discharge_voltage.png" alt="Cycle 1000" style="width: 100%; height: auto; border-radius: 8px;" /> <p style="margin-top: 12px; font-size: 15px; color: #555;"> Figure 2: Discharge Voltage curve at the beginning of life </p> </div> </div> As shown in Figures 1 and 2, the voltage curves have different lengths due to varying charge/discharge durations. Linear interpolation standardizes these curves to a fixed length (e.g., 500 points), enabling direct comparison across cycles as illustrated below figures 3 and 4

<div style="display: flex; flex-wrap: wrap; gap: 20px; justify-content: center; margin: 30px 0;"> <div style="flex: 1; min-width: 300px; max-width: 600px; text-align: center;"> <img src="charge_voltage_interpolated.png" alt="Cycle 1" style="width: 100%; height: auto; border-radius: 8px;" /> <p style="margin-top: 12px; font-size: 15px; color: #555;"> Figure 3: Inteporated Charging Voltage curve at the beginning of life </p> </div>

<div style="flex: 1; min-width: 300px; max-width: 600px; text-align: center;"> <img src="discharge_voltage_interpolated.png" alt="Cycle 1000" style="width: 100%; height: auto; border-radius: 8px;" /> <p style="margin-top: 12px; font-size: 15px; color: #555;"> Figure 4: Inteporated Discharge Voltage curve at the beginning of life </p> </div> </div>

2. Data Filtering: Cycles exhibiting non-meaningful behavior (e.g., flat voltage profiles, excessive noise, or unrealistic starting or peak voltages) are removed to ensure that all inputs correspond to valid charging or discharging events.


### Diagnostic Feature Extraction 

With aligned and cleaned curves, it is now possible to reliably extract cycle-specific diagnostic features that quantify the battery’s underlying physical degradation processes. These features are sensitive to aging mechanisms such as active material loss and internal resistance growth

| **Diagnostic Feature** | **Formula** | **Physical Meaning / Interpretation** |
| :--- | :--- | :--- |
| **Voltage Gap** | $\Delta \bar{V} = \bar{V}_c - \bar{V}_d$ | AAverage polarization; quantifies internal losses (overpotentials and resistance). |
| **Voltage Hysteresis** | $\Delta V(x) = V_c(x) - V_d(x)$ | Loss mechanisms at specific state-of-charge; reflects kinetic, ohmic, and diffusion effects. |
| **IC Peak Metrics** | $\text{IC} = \frac{dQ}{dV}$ | Phase transitions; peak shifts and magnitudes indicate active material loss. |
| **Hysteresis Proxy Resistance** | $R_{\text{proxy}} \propto \frac{\Delta V(x)}{I_{\text{diff}}}$ | Proxy for internal resistance growth due to side reactions. |


> **🧠 Reflection**:  We chose to derive these physically meaningful features instead of feeding the entire, aligned time-series data (Figures 3 & 4). Why are these manually engineered features often preferred in industrial applications? Consider the trade-offs in **model complexity**, **training speed**, and the crucial **interpretability** of the final Bayesian coefficients ($\beta$).



### Statistical Feature Aggregation 

The diagnostic signals derived above (e.g., incremental capacity curves) remain high-resolution time- or cycle-series data. To produce robust, concise, and comparable inputs for the Bayesian regression model, a final aggregation step is performed by extracting statistical moments from each diagnostic signal $s(x)$. This aggregation reduces hundreds of data points per cycle into a small number of highly informative scalar features.

| **Statistical Feature** | **Role in Degradation Modeling** |
| :--- | :--- |
| **Mean** | Captures the overall trend or shift of the diagnostic signal. |
| **Standard Deviation** | Measures variability and cycle-to-cycle noise. |
| **Skewness** | Indicates asymmetry or bias in the signal distribution. |
| **Kurtosis** | Quantifies the presence of extreme values or anomalies. |
| **RMS (Root-Mean-Square)** | Represents the overall magnitude and stress level of the signal. |
| **Entropy** |  Measures irregularity or disorder, often increasing with non-uniform degradation. |
| **Crest Factor** | Compares peak magnitude to average signal level, highlighting abnormal peaks. |
| **AUC (Area Under the Curve)** | Captures cumulative effects such as total energy loss or degradation trends. |

These statistical summaries (e.g., $\text{Mean}(\text{IC})$, $\text{Std}(\Delta V)$) form the input vector $\mathbf{x}$ in the linear predictor $\eta_i = \dots + \mathbf{x}_i^{\top} \boldsymbol{\beta}$. Such summaries of early-cycle behavior often preserve key degradation signatures while significantly reducing model complexity.

### Feature selection

After extracting a broad set of diagnostic and statistical features, feature selection is required. Using all available features can lead to overfitting, increased model complexity, and multicollinearity, which compromises interpretability of the Bayesian coefficients ($\boldsymbol{\beta}$). The final four features selected for regression ($\mathbf{x}$) are:
- charge_current_auc
- charge_current_mean
- discharge_voltage_auc
- discharge_voltage_crest


### Load pre-processed CALCE dataset

In [ ]:
FIGSHARE_DOWNLOAD_URL = "https://ndownloader.figshare.com/files/59415941"
features = ["charge_current_auc", "charge_current_mean", "discharge_voltage_crest", "discharge_voltage_auc"]
target = "capacity"
data = pd.read_parquet(FIGSHARE_DOWNLOAD_URL, engine="pyarrow", columns=["cycle"] + [target] + features)

In [ ]:
upper_bound, lower_bound = data[target].max(), data[target].min()

In [ ]:
df = data[data.CellType == "CS2"].copy()
test_df = df[df.BatteryID != "CALCE_CS2_38"]
train_df = df[df.BatteryID == "CALCE_CS2_38"]

### Model building: Generalizability Test
Before training, we carefully partition the data to test the model's ability to generalize beyond the specific unit it learned from. This setup simulates a crucial real-world scenario where a model trained on a few prototype units must make predictions for an entire new batch of batteries.The strategy is as follows:

1. Isolate Cell Type: We first filter the entire dataset (data) to focus only on a single type of chemistry, Cell Type "CS2". This ensures the training and testing sets share fundamental physical properties, making the test a fair assessment of individual variability, not chemistry differences.
2. Train on One Unit: We designate a single, representative unit, "CALCE_CS2_38", as our train_df. The model will learn its degradation rate ($\lambda$) and operational sensitivities ($\boldsymbol{\beta}$) entirely from this unit's history.
3. Validate on the Fleet: The remaining batteries of the same chemistry are assigned to the test_df.This setup is intentionally challenging. Our model's success will be measured by its ability to accurately predict the capacity fade of the batteries in test_df units it has never seen before—by relying solely on the general parameters learned from the single training unit. This is the ultimate test of the model's generalizability.

In [ ]:
plots = []
for feature_name in features:
    plot = scatter_plot(train_df, y_col=feature_name)
    plots.append(plot + labs(title=feature_name) + modern_theme(font_size=9))

gggrid(plots, ncol=2) + ggsize(650, 450)

The figure below plots the four selected features against cycle number for a representative battery. These plots empirically validate the selection process, as all four features exhibit clear, monotonic changes with cycling and, critically, show a distinct shift or acceleration in slope as the battery enters the failure state ($\text{SoH} \le 80\%$, shown in green). This strong visual correlation provides high confidence that these inputs will effectively drive the degradation component of our Bayesian model.

## Model Implementation and Inference

With our input data now rigorously cleaned, aligned, scaled, and reduced to the four most informative features ($\mathbf{X}$), we are ready to implement and fit our Bayesian regression. As defined in  [Beta Likelihood and Priors]() subsection, our model structure is a Bayesian Beta Regression. It combines our engineering knowledge (Priors) with the observed data (Likelihood) to model the bounded $\text{SoH}$ (scaled capacity $\mu_i$) via the Logit Link function.

We pass the four selected features: charge_current_auc, charge_current_mean, delta_voltage_variance, and discharge_voltage_crest, as the operational inputs (our $\mathbf{X}$ matrix) into the model definition.

In [ ]:
# from bayes.regression.beta_degradation import beta_regression_model
model, scaler = beta_regression_model(
    train_df, features, target=target, upper_bound=upper_bound, lower_bound=lower_bound
)
model

### Prior Predictive Check: Sanity Testing Our Assumptions

Following the rigorous Bayesian workflow, we begin with a crucial Prior Predictive Check (PPC). The PPC involves simulating data from the model using only the Prior Distributions, before seeing any observed data. This acts as a vital sanity test for our embedded engineering knowledge.

In [ ]:
with model:
    prior_pred = pm.sample_prior_predictive(samples=1000)

fig, ax = plt.subplots(figsize=(4, 1.8))
az.plot_ppc(prior_pred, group="prior", ax=ax)
plt.xlabel("Capacity")
plt.ylabel("Density");

The figure below, compare the predicted prior distribution (green line) against the observed data (blue line). This plot is essential for validating that our model's structural assumptions align with physical reality

In [ ]:
y_prior = prior_pred.prior["capacity_pred"].stack(sample=["chain", "draw"]).values
y_obs = train_df[target].values
post_mean = y_prior.mean()
n_draws = 500
rng = np.random.default_rng(42)
draw_idx = rng.choice(y_prior.shape[0], size=n_draws, replace=False)
y_prior_subset = y_prior[:, draw_idx].flatten()
df_prior = pd.DataFrame(
    {
        "SoH": np.concatenate([y_obs, y_prior_subset]),
        "type": ["observed"] * len(y_obs) + ["prior"] * len(y_prior_subset),
    }
)
plot_density(
    df_prior,
    x_col="SoH",
    color_col="type",
    x_label="Capacity",
    fig_size=(500, 400),
    title="Prior comparison",
    subtitle="Prior Predictive Check",
)

The Prior Predictive Check (PPC) confirms the model's structural integrity, aligning with our tight prior specifications:

1. High Confidence Start: The predicted capacity is concentrated in a tight, dominant peak around $\approx 1.0$. This is directly enforced by the highly informative precision prior $\phi \sim \text{Gamma}(100, 2.0)$ (mean $\phi=50$), which ensures the model begins its estimation with high certainty (low noise).
2. Realistic Degradation Envelope: The prior distribution is highly constrained, lacking extreme spread across the capacity range. This controlled shape results from setting the tight standard deviation ($\sigma=0.1$) on the degradation rate prior ($\lambda$), minimizing the probability of immediate, catastrophic failure and constraining the model to plausible degradation trajectories.
3. Multimodal Coverage: The prior successfully allocates mass across lower capacity states (e.g., $0.4$ and $0.7$), reflecting the influence of the remaining uncertainty in $\lambda$ and $\text{degr\_amp}$. This structurally acknowledges degradation and failure possibilities without over-predicting their frequency.
   
Conclusion: The final model structure is sound, respecting physical bounds and combining high data quality confidence with a constrained, realistic acknowledgment of the degradation process. The model is now robust and ready for MCMC sampling.

## Running Inference (MCMC Sampling)

With the model fully defined, our priors validated via the PPC, and the data features prepared, we move to the final computational step: approximating the posterior distribution. This is the heart of Bayesian inference: transforming the prior uncertainty into an informed posterior distribution using the evidence from the data.

In [ ]:
with model:
    idata = pm.sample(2000, tune=2000, target_accept=0.95, random_seed=42)


As discussed in [Part 1](https://sambaiga.github.io/blog/2025/10/bayesian-modelling-01.html), the MCMC process uses the No-U-Turn Sampler (NUTS) to explore the parameter space. The primary arguments guide this process:

- tune=2000: Specifies 2000 initial samples that are used solely to adapt the sampler's step size and are then discarded. A high tuning value is crucial for complex, highly curved posteriors (like those involving Beta distributions) to ensure stable exploration.
- draws=2000: Specifies 2000 final samples kept from the chain. These collected samples form the final Posterior Distribution for every model parameter ($\lambda$, $\beta$, $\phi$).
- target_accept=0.99: Forces the sampler to take smaller, more cautious steps. This high acceptance rate is necessary to avoid divergences in challenging models, ensuring a high-quality, accurate representation of the posterior distribution, though it increases computation time.
  
The resulting idata object now contains thousands of samples for every single model parameter, representing our comprehensive, uncertainty-quantified solution. The next step is to ensure these samples are reliable

### Model Diagnostics: Has the MCMC Converged?

After running the computationally intensive MCMC sampler, the very first step is to check the convergence diagnostics. The reliability of our final posterior estimates (our final answers) depends entirely on whether the Markov Chains successfully explored the entire parameter space. As discussed in [Part 1](), we focus on two key metrics shown in the summary table:

In [ ]:
vars = ["beta", "intercept", "lambda_rate", "phi", "degr_amp"]
data_summary = az.summary(idata, var_names=vars, kind="diagnostics")[["ess_bulk", "ess_tail", "r_hat"]]
GT(data_summary.reset_index()).tab_header(title="", subtitle="Diagnostics Summary").cols_label(
    {
        "ess_bulk": "ESS Bulk",
        "ess_tail": "ESS Tail.",
        "r_hat": "R-hat",
    }
)

All parameters show an $\hat{R}$ of exactly $1.0$, confirming the multiple Markov chains mixed exceptionally well and agree on the true posterior location. This indicates that the sampler thoroughly explored the parameter space without getting stuck in local modes. The high ESS values (ranging 4,126 to 5,915) confirm we collected a sufficient number of non-autocorrelated samples for every parameter, yielding highly reliable estimates for means and credible intervals.

With convergence confirmed, the sampled data is now a reliable representation of the Posterior Distribution. The next stage is to analyze these distributions to quantify the degradation process and the impact of our operational features.


### Analyzing the Posterior Distribution
With MCMC convergence successfully confirmed, we analyze the resulting Posterior Distribution to quantify our findings. The figure below displays the marginal Posterior Distribution (density, left) and the raw MCMC Trace Plot (right) for our core model parameters.

In [ ]:
az.plot_trace(idata, var_names=vars, compact=True);

## Interpreting the Posterior: Certainty, Rate, and Risk

The power of the Bayesian approach lies in its ability to quantify uncertainty for every parameter, moving beyond simple point estimates. The posterior summaries below allow us to identify reliable degradation drivers and estimate the full range of possible fade rates, which is crucial for managing risk in a battery fleet.

The analyze_parameter function below acts as a post-processing utility dedicated to generating publication-ready summary tables from the output of the MCMC sampling.

In [ ]:
def analyze_parameter(
    idata,
    parameter: str,
    features: list[str] | None = None,
    hdi_prob: float = 0.95,
    title: str = "Parameter Summary",
    subtitle: str | None = None,
) -> GT:
    """Generates a formatted summary table for a single parameter using Great Tables.

    This function extracts posterior summary statistics from ArviZ InferenceData and
    returns a beautifully styled table suitable for reports, notebooks, or publications.

    Args:
        idata: ArviZ InferenceData object containing posterior samples.
        parameter: Name of the parameter to summarize (e.g., "beta", "alpha", "sigma").
        features: List of feature names to label rows. Required and used only when
            ``parameter == "beta"``. Length must match the number of coefficients.
        hdi_prob: Highest density interval probability (default: 0.95).
        title: Main title for the table.
        subtitle: Optional subtitle. If None and parameter is "beta", defaults to
            "Beta coefficient analysis".

    Returns:
        A Great Tables (GT) object ready for display or further customization.

    Raises:
        ValueError: If ``features`` is provided for non-beta parameters or has wrong length.

    Example:
        >>> gt = analyze_parameter(idata, "beta", features=X.columns.tolist())
        >>> gt  # displays nicely in Jupyter
    """
    if features is not None and parameter != "beta":
        raise ValueError("`features` should only be provided when parameter == 'beta'")

    # Get summary statistics from ArviZ
    summary_df = az.summary(
        idata,
        var_names=[parameter],
        hdi_prob=hdi_prob,
        kind="stats",
        fmt="wide",
    ).reset_index(names="feature")

    # Assign meaningful feature names for beta coefficients
    if parameter == "beta":
        if features is None:
            raise ValueError("`features` must be provided when analyzing 'beta' parameter")
        if len(features) != len(summary_df):
            raise ValueError(
                f"Length of features ({len(features)}) must equal number of beta coefficients ({len(summary_df)})"
            )
        summary_df["feature"] = features

    conditions = [
        summary_df["hdi_2.5%"] > 0,  # Entire interval is positive
        summary_df["hdi_97.5%"] < 0,  # Entire interval is negative
    ]
    choices = ["Positive", "Negative"]
    summary_df["certainty"] = np.select(conditions, choices, default="Uncertain")

    # Set default subtitle for beta coefficients
    if subtitle is None and parameter == "beta":
        subtitle = "Beta coefficient analysis"

    gt_table = (
        GT(summary_df)
        .tab_header(
            title=md(f"**{title}**"),
            subtitle=md(subtitle) if subtitle else None,
        )
        .fmt_number(
            columns=["mean", "sd", "hdi_2.5%", "hdi_97.5%"],
            decimals=3,
        )
        .data_color(
            columns=["certainty"],
            palette=["#E1DFDD", "#F18F01", "#F18F01"],
            domain=["Uncertain", "Negative", "Positive"],
        )
        .cols_label(
            feature=md("**Feature**"),
            mean=md("**Mean**"),
            sd=md("**SD**"),
            **{"hdi_2.5%": md("**HDI 2.5%**")},
            **{"hdi_97.5%": md("**HDI 97.5%**")},
        )
        .cols_align(align="center", columns=["mean", "sd", "hdi_2.5%", "hdi_97.5%", "Certainty"])
        .tab_options(
            table_font_size="14px",
            heading_title_font_size="20px",
            heading_subtitle_font_size="16px",
            row_group_font_weight="bold",
        )
    )

    return gt_table

### Identifying Reliable Degradation Drivers ($\boldsymbol{\beta}$ Coefficients)

The $\boldsymbol{\beta}$ coefficients relate our features (like current and voltage metrics) to $\text{SoH}$ via the logit link function ($\log(\frac{\mu}{1 - \mu})$). We determine a predictor’s credibility by checking if its $95\%$ Highest Density Interval (HDI) crosses zero.

In [ ]:
analyze_parameter(idata, "beta", features=features)

The resulting summary above confirms that only one feature is a statistically reliable degradation driver at the $95\%$ confidence level: the discharge_voltage_crest factor. The discharge_voltage_crest has a 95% High Density Interval (HDI) that is entirely negative (from $-0.595$ to $-0.315$). This confirms, with high certainty, that an increase in this factor accelerates capacity fade.

The HDI for the other three features (charge_current_auc, charge_current_mean, and delta_voltage_variance) crosses zero (e.g., for charge_current_auc, the HDI is $[-0.421, 0.128]$). This means the model cannot rule out the possibility that their true effect is zero or even slightly positive. They are not statistically significant drivers.

Quantitative Impact: Focusing on the strongest driver, the mean coefficient for the discharge_voltage_crest factor is $\beta = -0.457$. We interpret this on the odds scale:$$\text{Odds Ratio} = \exp(-0.457) \approx 0.6$$This means that a one-unit increase in the crest factor is associated with the odds of high battery capacity decreasing by approximately $40\%$ ($\approx 1 - 0.615$).

Because all other features are not statistically supported, we can confidently focus our maintenance protocols on monitoring and controlling the discharge_voltage_crest factor. 

### Quantifying the Degradation Rate ($\lambda_{\text{rate}}$)

The $\lambda_{\text{rate}}$ parameter governs the speed of capacity fade. By using a less restrictive Lognormal prior ($\sigma=1.0$), the data was able to precisely estimate the actual degradation rate and quantify the remaining uncertainty.

In [ ]:
analyze_parameter(idata, "lambda_rate", title="Lambda Rate Summary", subtitle="Degradation rate parameter")

The resulting parameter summary provides a highly reliable estimate. The model's best estimate for the fade rate is $\mathbf{0.003}$ per unit of cycle data. This value is lower than the initial expected rate (from the prior centered around $0.01$). The small Standard Deviation ($\text{SD}=0.001$) and the narrow $95\%$ HDI ($[0.001, 0.006]$) demonstrate low remaining uncertainty. The MCMC has successfully used the data to achieve a highly precise estimate of the degradation speed.

###  Model Precision ($\phi$)
The $\phi$ parameter measures the model's precision, or how tightly the observed data clusters around the model's predicted mean after all factors are accounted for.

In [ ]:
analyze_parameter(idata, "phi", title="Phi Summary", subtitle="Phi  parameter")

The resulting posterior for $\phi$ shows a high degree of certainty, with a narrow $95\%$ HDI. The mean value of $91.255$ is the model's best estimate for the precision of the underlying process. This high mean value indicates that the residual variance (unexplained noise) in $\text{SoH}$ is very low. In practical terms, it means our combined model structure (the exponential term, $\lambda$, and the four operational features, $\boldsymbol{\beta}$) does a highly effective job of explaining the overall variability observed in the battery fleet. The tight HDI confirms that we have estimated this high level of precision with high certainty. 

### Analyzing the Posterior Distribution
Following parameter interpretation, the final diagnostic step is the Posterior Predictive Check (PPC). This step is essential for verifying that the model, using the now-informed posterior parameters, can generate synthetic data that closely resembles the actual observations. If the distributions of the synthetic and observed data align, we have high confidence that our model has captured the underlying data generating process.

We use PyMC's ``sample_posterior_predictive`` function to draw new samples from the likelihood distribution, using the parameters stored in the converged MCMC chains (idata).

In [ ]:
with model:
    post_pred = pm.sample_posterior_predictive(idata, var_names=["y_obs"], random_seed=42)

y_post = post_pred.posterior_predictive["y_obs"].stack(sample=["chain", "draw"]).values
# Mean of the posterior predictive
post_mean = y_post.mean()
n_draws = 500
rng = np.random.default_rng(42)
draw_idx = rng.choice(y_post.shape[0], size=n_draws, replace=False)
y_post_subset = y_post[:, draw_idx].flatten()
y_post_subset = y_post_subset * (upper_bound - lower_bound) + lower_bound


df_posterior = pd.DataFrame(
    {
        "SoH": np.concatenate([y_obs, y_post_subset]),
        "type": ["observed"] * len(y_obs) + ["posterior"] * len(y_post_subset),
    }
)

In [ ]:
df = pd.concat([df_prior, df_posterior])
plot_density(
    df,
    x_col="SoH",
    color_col="type",
    x_label="Capacity",
    fig_size=(500, 400),
    title="Prior and Posterior Comparison",
    subtitle="",
)

The figure above plots the observed capacity data against the synthetic data generated by the model. The plot shows the Posterior Predictive Distribution (blue) and the Observed Distribution (red) overlap significantly, particularly in the main body of the distribution (around 0.7). This strong visual alignment confirms that our complex Beta Regression model, incorporating the exponential fade and the four operational features, is an excellent fit for the underlying battery capacity data.

## Predict capacity for a new battery

Our final objective is to move from parameter estimation to practical Prognosis. We achieve this by generating a complete Posterior Predictive Distribution (PPD) for the capacity of a new or future battery state. This process leverages the full set of uncertainty-quantified model parameters ($\lambda_{\text{rate}}, \boldsymbol{\beta}, \text{and } \phi$). This process transforms our uncertainty-quantified parameter estimates into practical Prognostic predictions. This distribution automatically accounts for two critical types of uncertainty inherent in any prediction:
1. Epistemic Uncertainty: Uncertainty in the parameter estimates (e.g., how wide the HDI was for $\lambda_{\text{rate}}$).
2. Aleatoric Uncertainty: Inherent noise in the measurement process ($\phi$).

The output is not a single point, but a range ($\text{HDI}$) that tells us exactly where the true capacity is likely to fall.

To validate the model’s generalization capability and demonstrate its utility for risk management, we select three batteries from the CALCE dataset that were specifically excluded during model training. For each battery, we extract the same four features used in the original regression.



In [ ]:
from bayes.regression.beta_degradation import get_posterior_predictions

In [ ]:
new_df = test_df[test_df["BatteryID"].isin(["CALCE_CS2_34", "CALCE_CS2_36", "CALCE_CS2_37", "CALCE_CS2_33"])].copy()

The prediction process is summarized in the ``get_posterior_predictions`` function involves several key steps to generate predictions on new, unseen data using the fitted Bayesian model.:

- **Transform Input Features**: First, the new input feature data (data[features]) is transformed using the exact same scaler object that was fitted during the model training phase. This is crucial for ensuring the new data is on the same scale as the training data.

    ```python
    x_new = scaler.transform(data[features])
    ```
    Then we extracts the cycle numbers, which are a direct predictor variable required by the exponential degradation component in the Bayesian model.
   

- **Create Dummy Target Array**: The target variable container (y_obs in the PyMC model) must match the size of the new input data. We supply a dummy array (filled with an arbitrary value like $0.5$) solely to satisfy this structural size requirement, as its values are ignored during prediction.
   
   ```python
    y_dummy_scaled = np.full(shape=(X_new_scaled.shape[0],), fill_value=0.5)
   ```

- **Set New Data into Model**: Once all data arrays are prepared, the new feature data (x_new), cycle data (cycle_new), and the dummy target data (y_dummy_scaled) are set into the PyMC model's data containers using pm.set_data. This effectively prepares the model structure to run predictions on the new inputs.
  
   ```python
    with battery_model:
        pm.set_data({
            "x_data": x_new,
            "cycle_data": cycle_new,
            "y_obs": y_dummy_scaled
        })
   ```
- **Generate Posterior Predictive Samples**: Finally, we use the ```pm.sample_posterior_predictive``` function to generate predictions for the new battery data. This function draws samples from the Posterior Predictive Distribution (PPD), accounting for both the uncertainty in the model parameters (from idata) and the expected observation noise (from the Beta likelihood).
   
   ```python
    with battery_model:
        post_pred = pm.sample_posterior_predictive(
            idata,
            var_names=["y_obs"],
            random_seed=42,
            predictions=True,
        )
   ```


In [ ]:
pred_df = get_posterior_predictions(
    idata, model, scaler, new_df, features, alpha=0.1, upper_bound=upper_bound, lower_bound=lower_bound
)

We then visualize the predictions for each of the three test batteries using the ```plot_hdi_regression``` function. This function plots the mean predicted capacity curve along with the $95\%$ HDI, overlaid with the actual observed capacity data for comparison.

The plots below show the observed capacity data (small points) against the model's Posterior Predictive Distribution (shaded area).
The blue line represents the Posterior Predictive Mean (the model's best guess for the expected capacity at any given cycle), and the shaded area represents the $95\%$ HDI of the predicted capacity, effectively quantifying our uncertainty about the battery's future state.

In [ ]:
from bayes.plot.regres_plot import plot_hdi_regression

In [ ]:
plot_hdi_regression(
    pred_df,
    x_column="cycle",
    y_column="capacity",
    group_column="BatteryID",
    pred_column="pred_median",
    x_label="Cycle Number",
    y_label="Capacity (Ah)",
    title_prefix="Battery Capacity vs. Cycle with Posterior Predictions",
    subtitle="90% HDI accounts for  uncertainty.",
    alpha=0.1,
) + ggsize(600, 500)

Key Observations from the Plot
1. The blue mean line tracks the observed capacity data closely across the entire life of the battery, including the non-linear degradation near the end of life (EOL).
2. The vast majority (ideally $\approx 95\%$) of the observed capacity points should fall within the shaded prediction interval, demonstrating that the model's estimated uncertainty is reliable (high PICP).
3.  The HDI band appears relatively narrow during the stable phase of the battery.  For cells CS2_33, CS2_36, and CS2_37, the HDI band narrows or remains tight through the final, steep degradation phase, suggesting high model confidence and low noise even during non-linear behavior. However, for cell CS2_34, the HDI band visibly widens toward the final cycles, reflecting increased model uncertainty.


### Evaluating Predictive Performance

We will evaluate the model's performance on the test data using a comprehensive set of metrics:

1. Accuracy ($R^2$, MAE and RMSE): These metrics measure how close the point prediction is to the true capacity value. For instance high $R^2$ indicates the model's mean curve captures the overall trend (degradation slope) well.
2. Reliability (PICP and NMPI): Measures the quality and tightness of the predicted uncertainty range ($\text{HDI}$). PICP measures the percentage of the true capacity observations that fall within the predicted HDI. On the other hand, NMPI quantifies the average width of the HDI relative to the range of observed capacities. A low NMPI indicates a tight uncertainty range, which is desirable for precise risk management.


The following code calculates these metrics for each individual test battery:

In [ ]:
from bayes.metrics.interval import get_interval_metrics
from bayes.metrics.regression import regression_report

In [ ]:
metrics_list = []
for battery_id, df in pred_df.groupby("BatteryID"):
    reg_report = regression_report(df["capacity"], df["pred_median"])
    interval_report = get_interval_metrics(
        df["pred_median"].values,
        df["capacity"].values,
        df["hdi_low"].values,
        df["hdi_high"].values,
        alpha=0.1,
    )
    full_report = pd.concat([reg_report, interval_report], ignore_index=True)
    full_report["BatteryID"] = battery_id
    metrics_list.append(full_report)
metrics_df = pd.concat(metrics_list, ignore_index=True)

metrics_df = metrics_df.pivot_table(
    index=["BatteryID"],
    columns="Metric",
    values="Value",
).reset_index()

In [ ]:
def make_metrics_table(metrics_df, title="Model Evaluation Results"):
    """Generate a formatted GT table from cross-validation metrics."""
    # Sort to present best models first
    df = metrics_df.sort_values(["BatteryID", "MAE"]).reset_index(drop=True)

    # Build base table
    gt = (
        GT(df[["BatteryID", "MAE", "RMSE", "R2", "NMPI", "PICP"]])
        .tab_header(title=title, subtitle="Per-battery performance")
        .cols_label(BatteryID="Test Cell", MAE="MAE", RMSE="RMSE", R2=md("R<sup>2</sup>"))
        .fmt_number(columns=["MAE", "RMSE", "NMPI", "PICP"], decimals=3)
        .fmt_number(columns="R2", decimals=3)
        .tab_spanner(label="Error Metrics", columns=["MAE", "RMSE", "NMPI", "PICP"])
        .tab_style(
            style=style.text(weight="bold"),
            locations=loc.body(columns="Model"),
        )
        .tab_options(
            table_font_size="small",
            # row_strip_color="#fafafa"
        )
    )
    for col in ["MAE", "RMSE", "R2", "NMPI", "PICP"]:
        best_idx = df[col].idxmax() if col in ["R2", "PICP"] else df[col].idxmin()
        gt = gt.tab_style(style=style.fill(color="#00B294"), locations=loc.body(rows=best_idx, columns=col))

    return gt

In [ ]:
make_metrics_table(metrics_df, title="Model Evaluation Results")

The predictive performance of the Bayesian Beta Regression model is quantitatively evaluated on the held-out test batteries. The results confirm the visual success demonstrated in the preceding plots

1. Accuracy: TThe model achieves a near-perfect fit ($R^2$ between $0.940$ and $0.990$) with minimal absolute error ($\text{MAE}$ of $0.020$ to $0.030$ SoH units). This high accuracy is visually confirmed by the blue mean line tracking the observed data almost perfectly in the plots.
2. Sharpness ($\text{NMPI}$): The Normalized Mean Prediction Interval is consistently low ($\mathbf{0.180 – 0.200}$). This confirms that the prediction interval is significantly narrower than the full capacity range, quantifying the low uncertainty visually represented by the tight, thin bands around the mean curve.
3. Reliability ($\text{PICP}$): The model demonstrates perfect $100\%$ coverage ($\text{PICP}=1.000$) for all four tested cells. This verifies that the prediction intervals are well-calibrated and trustworthy, representing a conservative but highly reliable outcome.

## Conclusion

We have successfully built a powerful single-level Bayesian model that provides trustworthy uncertainty bounds. This model provides highly accurate point predictions and reliable $95\%$ credible intervals, validating the use of informed priors and the Beta regression structure for modeling capacity degradation.However, we modeled the degradation rate ($\lambda_{\text{rate}}$) and operational effects ($\boldsymbol{\beta}$) as fixed across the entire fleet. 

In reality, manufacturing variation (such as slight manufacturing defects or differences in batch chemistry) means that Battery A might degrade faster than Battery B under the same conditions. A 'one-size-fits-all' model, even a very accurate one like this, is inherently limited because it cannot learn the unique individual characteristics of each unit, leading to either under-predicted Risk  for batteries in a "bad batch or "Over-predicted Caution for batteries in a "good batch.

**Coming Next**: Addressing Heterogeneity with Hierarchical Models. In Part 3, we will tackle this fundamental limitation by introducing Hierarchical Bayesian Models (HBMs). We will learn how to build a model that simultaneously learns a robust "Global Trend" for the entire fleet while allowing each individual battery to have its own informed "Local Deviation" for parameters like its degradation rate ($\lambda_{\text{rate}}$) and its sensitivity to operational features ($\boldsymbol{\beta}$).  This approach will significantly improve risk management and EOL prediction for large, heterogeneous battery fleets.

## References

1. Zhang, H., Li, Y., Zheng, S. et al. [Battery lifetime prediction across diverse ageing conditions with inter-cell deep learning](https://www.nature.com/articles/s42256-024-00972-x#citeas). Nat Mach Intell 7, 270–277 (2025). https://doi.org/10.1038/s42256-024-00972-x
2. Ferrari, S. L. P., & Cribari-Neto, F. (2004). Beta regression for modelling rates and proportions. Journal of Applied Statistics, 31(7), 799–815.
3. Gelman, A., Carlin, J. B., Stern, H. S., Dunson, D. B., Vehtari, A., & Rubin, D. B. (2013). Bayesian Data Analysis (3rd ed.). CRC Press.
4. McElreath, R. (2020). Statistical Rethinking (2nd ed.). CRC Press.